## Chroma Import Update

In LangChain 1.x, the Chroma vector store integration is maintained in a separate package:

- Install: `pip install -U langchain-chroma`
- Import: `from langchain_chroma import Chroma`

This avoids the deprecation warning you saw when importing `Chroma` from `langchain_community.vectorstores`.ores.

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [2]:
loader_docx = Docx2txtLoader("Introduction_to_Data_and_Data_Science_2.docx")
pages = loader_docx.load()

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = [("#", "Course Title"), 
                           ("##", "Lecture Title")]
)

pages_md_split = md_splitter.split_text(pages[0].page_content)

for i in range(len(pages_md_split)):
    pages_md_split[i].page_content = ' '.join(pages_md_split[i].page_content.split())
    
char_splitter = CharacterTextSplitter(
    separator = ".",
    chunk_size = 500,
    chunk_overlap  = 50
)

pages_char_split = char_splitter.split_documents(pages_md_split)

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [3]:
len(pages_char_split)

20

In [6]:
pages_char_split[1] # this is a way print chunk files

Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past')

## Chroma Updating

In [11]:
#which place  save vector data (difine here)
vectorstore = Chroma.from_documents(documents=pages_char_split,embedding=embedding,persist_directory = "./intro-to-ds-lectures")

In [12]:
# Create (or load) a Chroma vector database stored on disk
# persist_directory tells Chroma where to save the vector data
# If the folder already exists, it loads existing embeddings
# If not, it creates a new vector database in that folder
vectorstore_from_directory = Chroma(persist_directory = "./intro-to-ds-lectures", 
                                    embedding_function = embedding)